# Presence Test

This notebook connects a second peer ("notebook-agent") back to itself via `runtimed`,
then animates cursor presence across cells. You should see a colored cursor bar and
gutter dots appear as the agent peer moves around.

**Requirements:** Open this notebook in the nteract dev app. The daemon will resolve
`runtimed` from `python/runtimed/pyproject.toml` automatically.

In [17]:
import time

import runtimed

In [18]:
# Connect a second peer to this same notebook.
# The daemon client auto-detects the socket path.
client = runtimed.DaemonClient()
rooms = client.list_rooms()
print(f"Open notebooks: {len(rooms)}")
for r in rooms:
    print(f"  {r['notebook_id']}")

Open notebooks: 1
  /Users/kylekelley/code/src/github.com/nteract/worktrees/desktop/provenance/pyt
hon/runtimed/demos/test-presence.ipynb


In [28]:
# Pick this notebook and connect as a second peer
notebook_id = None

for room in rooms:
    if "test-presence" in room["notebook_id"]:
        notebook_id = room["notebook_id"]

if notebook_id is None:
    raise ValueError("Test Presence not found")
session = runtimed.Session(notebook_id=notebook_id, peer_label="\U0001f916 Agent")
session.connect()
print(f"Connected to {notebook_id} as second peer")

Connected to /Users/kylekelley/code/src/github.com/nteract/worktrees/desktop/pro
venance/python/runtimed/demos/test-presence.ipynb as second peer


In [22]:
# Animate cursor across cells — watch the gutter dots jump
print("Jumping between cells...")
for c in cells:
    session.set_cursor(c.id, line=0, column=0)
    time.sleep(0.3)
print("Done — did you see the dots?")

Jumping between cells...
Done — did you see the dots?


In [27]:
# EDIT THIS CELL

print("hello")

hello


In [23]:
# Animate cursor within this cell — watch the cursor bar sweep
code_cells = [
    c
    for c in cells
    if c.cell_type == "code" and "Animate cursor within this cell" in c.source
]
if code_cells:
    target = code_cells[0]
    lines = target.source.split("\n")
    print(f"Sweeping cursor across '{lines[0][:30]}...' ({len(lines)} lines)")
    for line_num, line_text in enumerate(lines[:10]):
        for col in range(0, len(line_text) + 1, 2):
            session.set_cursor(target.id, line=line_num, column=col)
            time.sleep(0.01)
    print("Sweep complete")
else:
    print("No code cells with enough content to demo")

Sweeping cursor across '# Animate cursor within this c...' (17 lines)
Sweep complete


In [24]:
# Selection demo — grow a highlight across a cell

code_cells = [
    c for c in cells if c.cell_type == "code" and "Selection demo" in c.source
]

if code_cells:
    target = code_cells[0]
    lines = target.source.split("\n")
    print("Growing selection...")
    for line_num, line_text in enumerate(lines[:4]):
        for col in range(0, len(line_text) + 1, 3):
            session.set_selection(
                target.id,
                anchor_line=0,
                anchor_col=0,
                head_line=line_num,
                head_col=col,
            )
            time.sleep(0.05)
    time.sleep(1)
    # Clear by setting cursor
    # session.set_cursor(target.id, line=0, column=0)
    print("Selection cleared")

Growing selection...
Selection cleared


In [26]:
# Clean up — disconnect the second peer
session.close()
print("Second peer disconnected — cursor should disappear")

Second peer disconnected — cursor should disappear
